# Практикум 3. От результатов процедур к плану аудита (Google Colab)
**После блокнота вы умеете:** (1) сгруппировать находки по темам и назвать кластеры; (2) обучить модель риска на истории проверок со сплитом по времени и честно оценить её, включая смещение выборки; (3) откалибровать вероятности и проверить пользу решения по кривой чистой выгоды; (4) отобрать объекты под бюджет по ожидаемой ценности и разложить их в годовой план; (5) посчитать выборку МУС; (6) собрать черновик программы аудита с трассировкой к признакам.
**Данные:** синтетическая «вселенная аудита» (420 объектов × 5 лет) с известной истиной и тексты находок. Признак `anom_alerts` — число алертов из практикума 2.
**Артефакты:** `p3_metrics.json`, `audit_plan.csv`, `clusters.csv`, `audit_program_draft.md`, рисунки `fig_*.png`. Запуск: Среда выполнения → Выполнить всё.

In [ ]:
import numpy as np, pandas as pd, json, re
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, silhouette_score, adjusted_rand_score
from sklearn.inspection import permutation_importance
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
SEED = 42; rng = np.random.default_rng(SEED); pd.options.display.float_format = '{:,.3f}'.format
sig = lambda x: 1/(1+np.exp(-x)); z = lambda s: (s-s.mean())/s.std()
plt.rcParams.update({'font.size':10,'axes.spines.top':False,'axes.spines.right':False,'axes.titlesize':11})
TEAL, AMBER, RED, NAVY, GREY = '#0E7C86', '#F2A541', '#D1495B', '#0B2545', '#9AA5B1'

## 0. Вселенная аудита: объекты, годы, результаты проверок
Знаем истину для **всех** объектов (латентный `y_true`), но в истории видим результат только там, где проверка была (`audited`). Так в жизни: метки есть лишь по проверенному.

In [ ]:
N, YEARS = 420, [2021,2022,2023,2024,2025]
PROC = ['Платежи','Кредиты','Закупки','ИТ','Казначейство','Комплаенс','HR','Отчётность']
obj = pd.DataFrame({'obj_id': np.arange(N), 'unit': rng.choice([f'Подр.{i:02d}' for i in range(1,13)], N), 'process': rng.choice(PROC, N),
                    'base_turn': rng.lognormal(4,0.9,N), 'u': rng.normal(0,.6,N), 'it': rng.integers(1,9,N), 'aut': rng.beta(2,3,N)})
months = rng.integers(6,49,N).astype(float); parts = []
for yr in YEARS:
    d = obj[['obj_id','unit','process']].copy(); d['year'] = yr
    d['log_turnover'] = np.log(obj.base_turn)+.05*(yr-2021)+rng.normal(0,.05,N); zt = z(d.log_turnover)
    d['incidents'] = rng.poisson(np.exp(.5+.5*obj.u+.15*zt)); d['kri'] = rng.poisson(np.exp(.1+.5*obj.u)); d['anom_alerts'] = rng.poisson(np.exp(.1+.5*obj.u+.2*zt))
    d['months_since_audit'] = months.copy(); d['staff_turnover'] = np.clip(rng.beta(2,8,N)+.08*obj.u, 0, .6)
    d['process_change'] = rng.binomial(1,.2,N); d['reg_change'] = rng.binomial(1,.15,N); d['it_systems'] = obj.it; d['automation'] = obj.aut; d['complaints'] = rng.poisson(np.exp(.3*obj.u))
    d['logit0'] = (.6*zt+.4*z(d.incidents)+.3*z(d.kri)+.25*z(d.anom_alerts)+1.2*(1-np.exp(-d.months_since_audit/24))+2.5*(d.staff_turnover-.2)
                   +.6*d.process_change*(zt>0)+.3*d.reg_change*(d.it_systems>4)-.8*(d.automation-.4)+.9*obj.u+rng.normal(0,.3,N))
    d['audited'] = rng.random(N) < sig(-.5+.5*zt+.6*(d.months_since_audit/24-1))             # прежняя практика: проверяли крупное и давно не проверенное
    months = np.where(d.audited, 0, months)+12; parts.append(d)
df = pd.concat(parts, ignore_index=True)
b = brentq(lambda b: sig(df.logit0+b).mean()-.17, -10, 10); df['y_true'] = (rng.random(len(df)) < sig(df.logit0+b)).astype(int)
df['V'] = np.round(1.5*np.exp(.55*df.log_turnover)*rng.lognormal(0,.35,len(df)), 2)             # ценность выявления, млн
df['c_days'] = np.clip(np.round(8+2.5*df.it_systems+3*df.log_turnover+rng.normal(0,2,len(df))), 5, None)   # стоимость проверки, чел.-дней
print(df.shape, 'доля нарушений', df.y_true.mean().round(3), '| проверено', df.audited.mean().round(3))

## 1. Группировка находок: TF-IDF и кластеры
Тексты находок по проверенным объектам. Число кластеров выбираем по силуэту, качество проверяем по известным темам (ARI).

In [ ]:
TH = {'доступ': ['Выявлены избыточные права доступа сотрудников к системе {s}','Не проведена актуализация прав пользователей в {s}','Учётные записи уволенных сотрудников в {s} не заблокированы вовремя'],
      'сверка': ['Не выполнена ежедневная сверка остатков по счетам, выявлены расхождения','Расхождения между учётной системой и реестром операций не устранены в срок','Сверка по счёту проводится с нарушением периодичности'],
      'лимиты': ['Превышение установленных лимитов по операциям без согласования','Лимиты на операции не пересмотрены после изменения процесса','Операции сверх лимита, согласование оформлено задним числом'],
      'документы': ['Отсутствуют подтверждающие документы по операциям','Первичные документы оформлены с нарушением регламентов','Договор не содержит обязательных реквизитов, документы не подписаны'],
      'поставщики': ['Не проведена проверка контрагента {v} при заключении договора','Выбор поставщика выполнен без конкурентной процедуры','Выявлены признаки аффилированности с поставщиком {v}'],
      'данные': ['Выявлены ошибки в данных реестра {s}, влияющие на отчётность','Дублирование записей в справочнике контрагентов {s}','Отсутствует контроль полноты данных при выгрузке из {s}']}
FAV = {'Платежи':['сверка','лимиты'],'Кредиты':['документы','лимиты'],'Закупки':['поставщики','документы'],'ИТ':['доступ','данные'],'Казначейство':['лимиты','сверка'],'Комплаенс':['документы','поставщики'],'HR':['доступ','документы'],'Отчётность':['данные','сверка']}
SYS, VEN, TAIL = ['АБС','ERP','СЭД','CRM'], ['«Вектор»','«Север-Снаб»','«Альфа-Тех»'], ['', ' в ходе проверки', ' по состоянию на конец периода']
F = []
for r in df[df.audited].itertuples():
    for _ in range(rng.poisson(.8+2.2*r.y_true)):
        th = rng.choice(FAV[r.process]) if rng.random() < .7 else rng.choice(list(TH))
        F.append((r.obj_id, r.year, th, rng.choice(TH[th]).format(s=rng.choice(SYS), v=rng.choice(VEN))+rng.choice(TAIL)))
fd = pd.DataFrame(F, columns=['obj_id','year','theme_true','text']); tf = TfidfVectorizer(ngram_range=(1,2), min_df=2, max_df=.5); X = tf.fit_transform(fd.text)
XL = make_pipeline(TruncatedSVD(40, random_state=SEED), Normalizer(copy=False)).fit_transform(X)                 # LSA: убирает шум редких слов
fit_k = lambda k: KMeans(k, n_init=10, random_state=SEED).fit(XL)
sil = {k: silhouette_score(XL, fit_k(k).labels_) for k in range(4,13)}; K_sil = max(sil, key=sil.get); K = 6                  # K выбирает аналитик по смыслу
ARI = {k: round(adjusted_rand_score(fd.theme_true, fit_k(k).labels_),3) for k in (K_sil, K)}
km = fit_k(K); fd['cluster'] = km.labels_; terms = np.array(tf.get_feature_names_out())
cen = [np.asarray(X[km.labels_==c].mean(axis=0)).ravel() for c in range(K)]
top = {c: ', '.join(terms[np.argsort(-cen[c])[:3]]) for c in range(K)}
print('находок', len(fd), '| k по силуэту =', K_sil, '| ARI по k (силуэт, смысл):', ARI); print(top); fd.to_csv('clusters.csv', index=False)

## 2. Модель риска: сплит по времени, калибровка
Обучение — проверенные объекты 2021–2023, калибровка — 2024, тест — 2025. Тест смотрим дважды: на проверенных (как видит практика) и на **всех** объектах (истина).

In [ ]:
FE = ['log_turnover','incidents','kri','anom_alerts','months_since_audit','staff_turnover','process_change','reg_change','it_systems','automation','complaints']
MONO = [1,1,1,1,1,1,1,0,0,-1,1]                                              # допустимое направление влияния
tr = df[df.audited & (df.year<=2023)]; ca = df[df.audited & (df.year==2024)]; te = df[df.year==2025]; te_a = te[te.audited]
def ece(y,p,bins=10):
    q = pd.qcut(p, bins, duplicates='drop'); g = pd.DataFrame({'y':y,'p':p,'q':q}).groupby('q', observed=True)
    return float((g.size()/len(y)*(g.y.mean()-g.p.mean()).abs()).sum())
def ev(y,p): return {'ROC-AUC': roc_auc_score(y,p), 'PR-AUC': average_precision_score(y,p), 'Brier': brier_score_loss(y,p), 'ECE': ece(y,p),
                     'Lift@10%': float(y[np.argsort(-p)[:max(1,len(y)//10)]].mean()/y.mean())}
M = {'логрегрессия': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(tr[FE], tr.y_true),
     'GBM': HistGradientBoostingClassifier(max_depth=3, learning_rate=.05, max_iter=300, l2_regularization=1., random_state=SEED).fit(tr[FE], tr.y_true),
     'GBM + монотонность': HistGradientBoostingClassifier(max_depth=3, learning_rate=.05, max_iter=300, l2_regularization=1., monotonic_cst=MONO, random_state=SEED).fit(tr[FE], tr.y_true)}
lgt = lambda p: np.log(np.clip(p,1e-4,1-1e-4)/(1-np.clip(p,1e-4,1-1e-4))).reshape(-1,1)
def platt(p_cal, y_cal): lr = LogisticRegression(C=1e6).fit(lgt(p_cal), y_cal); return lambda p: lr.predict_proba(lgt(p))[:,1]       # сигмоидная калибровка (Platt)
cal = {k: platt(m.predict_proba(ca[FE])[:,1], ca.y_true) for k,m in M.items()}
iso = {k: IsotonicRegression(out_of_bounds='clip').fit(m.predict_proba(ca[FE])[:,1], ca.y_true) for k,m in M.items()}
P = {k: cal[k](m.predict_proba(te[FE])[:,1]) for k,m in M.items()}; Praw = {k: m.predict_proba(te[FE])[:,1] for k,m in M.items()}
R_all = pd.DataFrame({k: ev(te.y_true.values, P[k]) for k in M}).T; R_aud = pd.DataFrame({k: ev(te_a.y_true.values, P[k][te.audited.values]) for k in M}).T
print('ТЕСТ 2025, все объекты (истина)'); print(R_all); print('\nТЕСТ 2025, только проверенные (как видит практика)'); print(R_aud)
Pi = iso['GBM'].predict(Praw['GBM']); print('\nGBM: ECE до калибровки %.3f, Platt %.3f, изотоническая %.3f | уникальных p: Platt %d, изотоническая %d' % (ece(te.y_true.values, Praw['GBM']), R_all.loc['GBM','ECE'], ece(te.y_true.values, Pi), len(np.unique(P['GBM'].round(4))), len(np.unique(Pi.round(4)))))
print('Базовая практика: AUC по возрасту проверки %.3f, по обороту %.3f' % (roc_auc_score(te.y_true, te.months_since_audit), roc_auc_score(te.y_true, te.log_turnover)))

In [ ]:
best = max(M, key=lambda k: average_precision_score(ca.y_true, M[k].predict_proba(ca[FE])[:,1])); print('лучшая по валидации 2024:', best); pi = permutation_importance(M[best], te[FE], te.y_true, scoring='average_precision', n_repeats=10, random_state=SEED)
IMP = pd.Series(pi.importances_mean, FE).sort_values(ascending=False); print(IMP.round(3).head(6))
def net_benefit(y, p, t): tp = ((p>=t)&(y==1)).sum(); fp = ((p>=t)&(y==0)).sum(); return tp/len(y)-fp/len(y)*t/(1-t)
TH_ = np.linspace(.05,.5,46); yt = te.y_true.values; print('чистая выгода при t=0.2: модель %.3f, проверять всех %.3f' % (net_benefit(yt,P[best],.2), net_benefit(yt,np.ones(len(yt)),.2)))
rs = np.random.default_rng(1); LB = [(lambda ix: yt[ix][np.argsort(-P[best][ix])[:len(ix)//10]].mean()/yt[ix].mean())(rs.integers(0,len(yt),len(yt))) for _ in range(1000)]
print('Lift@10%% бутстреп 95%%: %.2f–%.2f' % tuple(np.percentile(LB,[2.5,97.5])))

## 3. Приоритизация под бюджет
`EV = p·V − c·ставка`, отбор по `EV/c` жадным способом до исчерпания бюджета в человеко-днях. Сравниваем со стратегиями «по возрасту проверки», «по обороту», «по p». Реализованная ценность считается по истине `y_true`.

In [ ]:
RATE, B = 0.06, 2000                                                    # млн за чел.-день; бюджет чел.-дней
t = te.reset_index(drop=True).copy(); t['p'] = P[best]; t['EV'] = t.p*t.V-RATE*t.c_days
def pick(order, budget=B):
    sel, used = [], 0
    for i in order:
        if used+t.c_days.iloc[i] <= budget: sel.append(i); used += t.c_days.iloc[i]
    return sel
t['p_true'] = sig(te.logit0.values+b)                                          # истинная вероятность (знаем только в синтетике)
def realized(sel): s = t.iloc[sel]; return {'объектов': len(sel), 'находок': int(s.y_true.sum()), 'ценность факт, млн': float((s.y_true*s.V-RATE*s.c_days).sum()), 'ценность матожид., млн': float((s.p_true*s.V-RATE*s.c_days).sum())}
O = {'по возрасту проверки': np.argsort(-t.months_since_audit.values), 'по обороту': np.argsort(-t.log_turnover.values), 'по p': np.argsort(-t.p.values),
     'по EV/c (бюджет)': np.argsort(-(t.EV/t.c_days).values), 'случайно': rng.permutation(len(t))}
ST = pd.DataFrame({k: realized(pick(o)) for k,o in O.items()}).T; print(ST)
sel = pick(O['по EV/c (бюджет)']); rest = [i for i in range(len(t)) if i not in sel]; expl = list(rng.choice(rest, 6, replace=False))
print('exploration: +6 случайных объектов из непроверенных для свежих меток; доля бюджета ≈ %.0f%%' % (100*t.c_days.iloc[expl].sum()/B))

In [ ]:
plan = t.iloc[sel][['obj_id','unit','process','p','V','c_days','EV']].copy(); plan['priority'] = np.arange(1,len(plan)+1)
free = np.zeros(4); st, tm = [], []                                    # 4 команды, длительность = дни / 10 (два аудитора)
for d_ in plan.c_days: k = int(np.argmin(free)); st.append(free[k]); tm.append(k); free[k] += d_/10
plan['team'] = np.array(tm)+1; plan['start_week'] = st; plan['weeks'] = plan.c_days/10; plan['quarter'] = (plan.start_week//13).astype(int).clip(0,3)+1
print(plan.groupby('quarter').size().to_dict(), '| недель по командам', free.astype(int).tolist()); plan.to_csv('audit_plan.csv', index=False)
BV = float(np.round(rng.lognormal(8,1.1,5000),2).sum()); TM, EM, RF, EF = .02*BV, .005*BV, 3.0, 1.6
n_mus = int(np.ceil(BV*RF/(TM-EM*EF))); print('Размер выборки МУС: n = BV·RF/(TM − EM·EF) =', n_mus)
def mus_sample(vals, n, seed=0):
    r = np.random.default_rng(seed); cum = np.cumsum(vals); step = cum[-1]/n; pts = r.uniform(0,step)+step*np.arange(n); return np.unique(np.searchsorted(cum, pts))
pop = np.round(np.random.default_rng(7).lognormal(8,1.1,5000),2); smp = mus_sample(pop, n_mus); print('отобрано позиций', len(smp), '| доля стоимости в выборке', round(pop[smp].sum()/pop.sum(),3))

## 4. Черновик программы аудита с трассировкой
Причины — признаки с наибольшим вкладом в риск (отклонение от среднего с учётом направления). Процедуры берём из **библиотеки**, LLM только переформулирует, а не придумывает.

In [ ]:
LIB = {'staff_turnover': ['Проверить передачу дел и актуализацию прав при смене сотрудников','Сверить права доступа со штатным расписанием'],
       'months_since_audit': ['Охватить процедуры, не проверявшиеся более 2 лет','Проверить устранение замечаний прошлых проверок'],
       'incidents': ['Проанализировать инциденты за период и их первопричины','Проверить полноту регистрации инцидентов'],
       'kri': ['Проверить причины выходов KRI за пороги и реакцию владельца процесса','Сверить расчёт KRI с первичными данными'],
       'anom_alerts': ['Разобрать очередь алертов практикума 2 по объекту','Проверить выборку проводок из очереди с причинами срабатывания'],
       'process_change': ['Проверить внедрение изменений процесса и обновление регламентов','Оценить актуальность контролей после изменений'],
       'automation': ['Проверить ручные операции и контроли их выполнения','Сверить ручные корректировки с первичными документами'],
       'complaints': ['Проанализировать жалобы клиентов и сроки реагирования','Проверить корректность закрытия обращений'],
       'log_turnover': ['Увеличить объём выборки по крупным операциям (МУС)','Проверить лимиты и согласования по крупным суммам'],
       'reg_change': ['Проверить соответствие процесса новым требованиям регулятора'], 'it_systems': ['Проверить интеграции и сверку данных между системами']}
DIR = dict(zip(FE, MONO)); mu, sd = tr[FE].mean(), tr[FE].std()
def drivers(row, top=3): zz = ((row[FE]-mu)/sd)*pd.Series(DIR).replace(0,1).values; return zz.sort_values(ascending=False).head(top)
o = t.sort_values('EV', ascending=False).iloc[0]; dr = drivers(o); L = [f'# Черновик программы аудита: объект {int(o.obj_id)} ({o.unit}, {o.process})', f'Оценка риска p = {o.p:.2f}; ожидаемая ценность {o.EV:.1f} млн; трудоёмкость {int(o.c_days)} чел.-дней.\n',
     '| Фактор риска | Значение | Процедура | Источник |', '|---|---|---|---|']
for f_, zv in dr.items():
    for pr in LIB.get(f_, [])[:2]: L.append(f'| {f_} (вклад {zv:+.1f}σ) | {o[f_]:.2f} | {pr} | поле {f_}, год {int(o.year)} |')
open('audit_program_draft.md','w').write('\n'.join(L)); print('\n'.join(L))
PROMPT = ('Роль: помощник аудитора. Перепиши процедуры ниже в формулировках программы аудита. Используй ТОЛЬКО перечисленные процедуры и факторы, не добавляй ссылок на нормативные акты и новых процедур. '
          'Для каждой строки сохрани колонку «Источник» без изменений.\n'+'\n'.join(L[4:])); print('\nДлина промпта:', len(PROMPT), 'символов')

## 5. Рисунки для презентации

In [ ]:
def style(ax): ax.grid(alpha=.25, lw=.5); ax.set_axisbelow(True)
# 1. Надёжность (калибровка)
fig, ax = plt.subplots(figsize=(5.6,3.9)); ax.plot([0,.8],[0,.8],'--',c=GREY,lw=1,label='идеал')
for lab,pp,c in [('до калибровки', Praw[best], AMBER), ('после (Platt)', P[best], TEAL)]:
    q = pd.qcut(pp, 8, duplicates='drop'); g = pd.DataFrame({'p':pp,'y':te.y_true.values,'q':q}).groupby('q', observed=True).mean(); ax.plot(g.p, g.y, 'o-', c=c, label=lab)
ax.set_xlabel('предсказанная вероятность'); ax.set_ylabel('доля нарушений'); ax.set_title('Диаграмма надёжности, тест 2025'); ax.legend(frameon=False); style(ax); fig.tight_layout(); fig.savefig('fig_reliability.png', dpi=200)
# 2. Накопленный выигрыш
fig, ax = plt.subplots(figsize=(5.6,3.9)); n_ = len(te)
for lab,sc_,c in [(best, P[best], TEAL), ('GBM' if best!='GBM' else 'логрегрессия', P['GBM' if best!='GBM' else 'логрегрессия'], '#5B8DB8'), ('по возрасту проверки', te.months_since_audit.values, AMBER)]:
    o_ = np.argsort(-sc_); ax.plot(np.arange(1,n_+1)/n_, np.cumsum(te.y_true.values[o_])/te.y_true.sum(), c=c, label=lab, lw=2)
ax.plot([0,1],[0,1],'--',c=GREY,label='случайно'); ax.set_xlabel('доля просмотренных объектов'); ax.set_ylabel('доля найденных нарушений'); ax.set_title('Накопленный выигрыш'); ax.legend(frameon=False); style(ax); fig.tight_layout(); fig.savefig('fig_gain.png', dpi=200)
# 3. Кривая чистой выгоды
fig, ax = plt.subplots(figsize=(5.6,3.9))
ax.plot(TH_, [net_benefit(yt,P[best],x) for x in TH_], c=TEAL, lw=2, label='модель: '+best); ax.plot(TH_, [net_benefit(yt,P['GBM' if best!='GBM' else 'логрегрессия'],x) for x in TH_], c='#5B8DB8', lw=1.5, label='GBM' if best!='GBM' else 'логрегрессия')
ax.plot(TH_, [net_benefit(yt,np.ones(len(yt)),x) for x in TH_], c=AMBER, lw=1.5, label='проверять всех'); ax.axhline(0, c=GREY, ls='--', label='не проверять')
ax.set_ylim(-.05, .2); ax.set_xlabel('порог вероятности'); ax.set_ylabel('чистая выгода'); ax.set_title('Анализ кривой решений'); ax.legend(frameon=False); style(ax); fig.tight_layout(); fig.savefig('fig_decision_curve.png', dpi=200)
# 4. Риск-карта
fig, ax = plt.subplots(figsize=(6.0,3.95)); s_ = t.index.isin(t.index[sel])
ax.scatter(t.p[~s_], t.V[~s_], s=t.c_days[~s_]*1.6, c=GREY, alpha=.45, label='вне плана'); ax.scatter(t.p[s_], t.V[s_], s=t.c_days[s_]*1.6, c=TEAL, alpha=.8, label='в плане')
ax.set_yscale('log'); ax.set_xlabel('вероятность существенной находки p'); ax.set_ylabel('ценность выявления V, млн'); ax.set_title('Риск-карта объектов (размер = трудоёмкость)'); ax.legend(frameon=False, loc='lower right'); style(ax); fig.tight_layout(); fig.savefig('fig_riskmap.png', dpi=200)
# 5. Кластеры находок
Z2 = TruncatedSVD(2, random_state=SEED).fit_transform(X); fig, ax = plt.subplots(figsize=(5.6,3.9)); cm = plt.cm.tab10(np.arange(K))
for c in range(K): m_ = fd.cluster.values==c; ax.scatter(Z2[m_,0], Z2[m_,1], s=10, color=cm[c], alpha=.6); ax.annotate(f'{c+1}', Z2[m_].mean(0), fontsize=12, fontweight='bold', ha='center')
ax.set_title('Карта находок: %d кластеров' % K); ax.set_xlabel('компонента 1'); ax.set_ylabel('компонента 2'); style(ax); fig.tight_layout(); fig.savefig('fig_clusters.png', dpi=200)
# 6. Бюджет
fig, ax = plt.subplots(figsize=(9.0,2.9)); od = O['по EV/c (бюджет)'][:120]; r_ = (t.EV/t.c_days).values[od]; colr = [TEAL if i in set(sel) else GREY for i in od]
ax.bar(range(len(od)), r_, color=colr, width=.85); ax.set_ylabel('EV на чел.-день, млн'); ax2 = ax.twinx(); ax2.plot(np.cumsum(t.c_days.values[od]), alpha=0); ax2.remove()
cut = max(i for i,x in enumerate(od) if x in set(sel)); ax.axvline(cut+.5, c=RED, ls='--'); ax.text(cut+1.5, r_.max()*.85, 'граница бюджета %d чел.-дн.' % B, color=RED); ax.set_xlabel('ранг объекта'); ax.set_title('Отбор по EV/c до исчерпания бюджета'); style(ax); fig.tight_layout(); fig.savefig('fig_budget.png', dpi=200)
# 7. Гант
fig, ax = plt.subplots(figsize=(9.0,3.9)); pr = plan.priority.values; col = np.where(pr<=len(plan)/3, TEAL, np.where(pr<=2*len(plan)/3, AMBER, GREY))
for r in plan.itertuples(): ax.barh(r.team, r.weeks, left=r.start_week, color=col[r.priority-1], edgecolor='white', height=.7)
ax.set_yticks([1,2,3,4]); ax.set_yticklabels([f'Команда {i}' for i in [1,2,3,4]]); ax.invert_yaxis(); ax.set_xlim(0,52)
for q_ in [13,26,39]: ax.axvline(q_, c=GREY, lw=.8, ls=':')
ax.set_xticks([6.5,19.5,32.5,45.5]); ax.set_xticklabels(['Q1','Q2','Q3','Q4']); ax.set_title('Годовой план: бирюзовый — верхняя треть приоритета'); fig.tight_layout(); fig.savefig('fig_gantt.png', dpi=200)

In [ ]:
json.dump({'test_all': R_all.to_dict('index'), 'test_audited': R_aud.to_dict('index'), 'strategies': ST.to_dict('index'), 'importance_top6': IMP.head(6).round(4).to_dict(),
           'lift10_ci': [float(x) for x in np.percentile(LB,[2.5,97.5])], 'clusters': {'k': int(K), 'k_silhouette': int(K_sil), 'ARI': ARI, 'terms': top},
           'mus_n': n_mus, 'plan_objects': int(len(plan))}, open('p3_metrics.json','w'), indent=1, ensure_ascii=False)

## Задания
1. Уберите `anom_alerts` и `months_since_audit`: как меняются PR-AUC и Lift@10% на всех объектах?
2. Обучите модель на **всех** объектах (используя `y_true`) и сравните с обучением на проверенных: насколько велико смещение выборки?
3. Задайте бюджет 1000 и 3000 чел.-дней: как меняется отбор и реализованная ценность?
4. Измените допустимую ошибку TM в МУС с 2% до 1%: во сколько раз растёт выборка?
5. Проверьте 3 строки черновика программы: все ли процедуры есть в библиотеке и сопровождаются источником?